# 🚀 ACE Pretraining Launcher

Launch ACE pretraining on **Google Colab** or **Kaggle** with GPU.

This notebook:
1. Clones the ACE repo and installs dependencies
2. Mounts Google Drive for persistent checkpoint storage
3. Configures and launches `scripts/pretrain.py`
4. Supports resuming from checkpoints

> **Requirements**: A Colab/Kaggle GPU runtime (T4, A100, etc.)

## 1. Environment Setup

In [ ]:
# ── Clone repo (replace with your repo URL) ──────────────────────────
import os
from pathlib import Path

REPO_URL = "https://github.com/YOUR_USERNAME/ACE.git"  # ← Update this
REPO_DIR = Path("/content/ACE")

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
    print(f"Cloned repo to {REPO_DIR}")
else:
    !cd {REPO_DIR} && git pull
    print(f"Repo already exists, pulled latest")

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

In [ ]:
# ── Install dependencies ──────────────────────────────────────────────
!pip install -e . -q
!pip install wandb tqdm -q

# Verify GPU
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# ── Mount Google Drive (Colab only) ───────────────────────────────────
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_CHECKPOINT_DIR = "/content/drive/MyDrive/ace_checkpoints"
    Path(DRIVE_CHECKPOINT_DIR).mkdir(parents=True, exist_ok=True)
    print(f"Drive mounted. Checkpoints will be saved to: {DRIVE_CHECKPOINT_DIR}")
except ImportError:
    # Not on Colab (Kaggle or local)
    DRIVE_CHECKPOINT_DIR = "/content/ACE/checkpoints"
    Path(DRIVE_CHECKPOINT_DIR).mkdir(parents=True, exist_ok=True)
    print(f"Not on Colab. Checkpoints: {DRIVE_CHECKPOINT_DIR}")

## 2. Configuration

In [ ]:
# ── Training Configuration ────────────────────────────────────────────

# Model config YAML
CONFIG_FILE = "configs/ace_micro.yaml"

# Training hyperparameters
BATCH_SIZE = 4
GRADIENT_ACCUMULATION = 4
MAX_STEPS = 10_000
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 0.1
WARMUP_STEPS = 500
GRADIENT_CLIP = 1.0
SEQ_LEN = 512

# Logging
USE_WANDB = False  # Set True and login first: !wandb login
LOG_EVERY = 10
CHECKPOINT_EVERY = 500

# Resume (set to checkpoint dir path to resume, or None)
RESUME_FROM = None  # e.g., DRIVE_CHECKPOINT_DIR

print("Configuration ready.")
print(f"  Config file:    {CONFIG_FILE}")
print(f"  Batch size:     {BATCH_SIZE} x {GRADIENT_ACCUMULATION} accum = {BATCH_SIZE * GRADIENT_ACCUMULATION} effective")
print(f"  Max steps:      {MAX_STEPS:,}")
print(f"  LR:             {LEARNING_RATE}")
print(f"  Checkpoint dir: {DRIVE_CHECKPOINT_DIR}")

In [ ]:
# ── Preview model config ──────────────────────────────────────────────
from ace.model.config import AceConfig

cfg = AceConfig.from_yaml(CONFIG_FILE)
print(cfg.summary())

## 3. Launch Training

In [ ]:
# ── Build the pretrain command ────────────────────────────────────────
cmd = (
    f"python scripts/pretrain.py"
    f" --config {CONFIG_FILE}"
    f" --batch_size {BATCH_SIZE}"
    f" --gradient_accumulation_steps {GRADIENT_ACCUMULATION}"
    f" --max_steps {MAX_STEPS}"
    f" --lr {LEARNING_RATE}"
    f" --weight_decay {WEIGHT_DECAY}"
    f" --warmup_steps {WARMUP_STEPS}"
    f" --gradient_clip {GRADIENT_CLIP}"
    f" --seq_len {SEQ_LEN}"
    f" --checkpoint_dir {DRIVE_CHECKPOINT_DIR}"
    f" --checkpoint_every {CHECKPOINT_EVERY}"
    f" --log_every {LOG_EVERY}"
    f" --bf16"
)

if USE_WANDB:
    cmd += " --use_wandb"

if RESUME_FROM is not None:
    cmd += f" --resume_from_checkpoint {RESUME_FROM}"

print("Command:")
print(cmd)
print()

# ── Run training ──────────────────────────────────────────────────────
!{cmd}

## 4. Monitor & Resume

In [ ]:
# ── List saved checkpoints ────────────────────────────────────────────
from ace.training.checkpoint import CheckpointManager

mgr = CheckpointManager(DRIVE_CHECKPOINT_DIR)
ckpts = mgr.list_checkpoints()

print(f"Found {len(ckpts)} checkpoint(s):")
for c in ckpts:
    print(f"  Step {c['step']:>8d}  |  {c['path'].name}  |  {c['timestamp']}")

In [ ]:
# ── Resume training (re-run after updating RESUME_FROM above) ─────────
# To resume:
#   1. Set RESUME_FROM = DRIVE_CHECKPOINT_DIR in the Configuration cell
#   2. Optionally increase MAX_STEPS
#   3. Re-run the "Launch Training" cell

print("To resume training:")
print(f"  1. Set RESUME_FROM = '{DRIVE_CHECKPOINT_DIR}'")
print(f"  2. Re-run cells 2 (Configuration) and 3 (Launch Training)")

In [ ]:
# ── Plot training loss (if logs were saved) ───────────────────────────
# If using W&B, view at: https://wandb.ai
# Otherwise, parse the training logs manually:

print("If using W&B, view your dashboard at https://wandb.ai")
print("Otherwise, check the terminal output above for loss curves.")